# 6.1. Перенос импульса по кольцу

Рассмотрим примесь, которая движется по кольцу единичной длины с постоянной скоростью $c=1$. Её концентрация $u(x,t)$ удовлетворяет уравнению
$$u_t+c u_x=0,\qquad 0\le x<1,\qquad u(x+1,t)=u(x,t).$$
Диффузии и источников в этой модели нет. Вдоль движущейся точки $x(t)=\xi+ct$ получаем
$$\frac{d}{dt}u(x(t),t)=u_t+c u_x=0,$$
поэтому точное решение — сдвиг начального профиля:
$$u(x,t)=u_0\bigl((x-ct)\bmod1\bigr).$$
За время $T=1$ примесь совершает один оборот. Форма профиля, его высота и количество вещества при этом сохраняются. После любого целого числа оборотов распределение совпадает с начальным.

Вычислим перенос схемой «против потока» (upwind) и схемой Лакса–Вендроффа. На гладкой синусоиде проверим порядок сходимости после одного и пяти оборотов. Затем используем прямоугольный импульс: на его фронтах особенно хорошо видны размытие и паразитные колебания. Сравним числа Куранта $C=0.5$, $0.8$, $1$ и отдельно исследуем нарушение устойчивости при $C=1.1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.ticker import NullFormatter
from IPython.display import display, Markdown, HTML

plt.rcParams.update({
    'figure.dpi': 115,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'animation.embed_limit': 60,
})

LABELS = {'upwind': 'Против потока', 'LW': 'Лакса–Вендроффа'}
STYLES = {
    'upwind': dict(color='#0072B2', lw=3.0, ls='-', zorder=3),
    'LW': dict(color='#D55E00', lw=1.9, ls=(0, (5, 3)), zorder=4),
}


def table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |',
             '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines += ['| ' + ' | '.join(map(str, row)) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def plot_numerical(ax, x, u, method, pulse_profile=False):
    # Точка x=1 добавляется только на рисунке, чтобы замкнуть кольцо.
    xx, yy = np.r_[x, 1.0], np.r_[u, u[0]]
    options = dict(**STYLES[method], label=LABELS[method])
    if pulse_profile:
        return ax.step(xx, yy, where='post', **options)[0]
    return ax.plot(xx, yy, **options)[0]


def exact_pulse_line(t):
    """Точные положения фронтов, в том числе при переходе через x=1."""
    edges = np.remainder(np.array([0.2, 0.4]) + t, 1.0)
    edges[np.minimum(abs(edges), abs(edges - 1)) < 1e-13] = 0
    boundaries = np.unique(np.r_[0.0, edges, 1.0])
    middle = (boundaries[:-1] + boundaries[1:]) / 2
    values = exact(middle, t, 'pulse')
    xx = np.column_stack((boundaries[:-1], boundaries[1:])).ravel()
    return xx, np.repeat(values, 2)


def plot_reference(ax, t, profile):
    if profile == 'pulse':
        xx, yy = exact_pulse_line(t)
    else:
        xx = np.linspace(0, 1, 1001)
        yy = exact(xx, t, profile)
    return ax.plot(xx, yy, color='black', ls=(0, (1.2, 3.0)), lw=1.2,
                   zorder=5, label='Точное решение')[0]


def levels(ax):
    ax.axhline(0, color='0.65', lw=0.8, zorder=1)
    ax.axhline(1, color='0.65', lw=0.8, zorder=1)

## 1. Как аппроксимируется перенос

Разобьём кольцо на $N$ одинаковых интервалов длины $h=1/N$. Используем узлы $x_j=jh$, $j=0,\ldots,N-1$; точку $x=1$ не включаем, поскольку она совпадает с $x=0$. Все индексы периодические: $U_{-1}=U_{N-1}$, $U_N=U_0$. Величина $U_j^n$ приближает значение концентрации в узле $x_j$ в момент $t_n$.

### Схема против потока

При $c>0$ информация приходит слева. Заменим производную по времени прямой разностью, а производную по пространству — разностью с левым соседом:
$$\frac{U_j^{n+1}-U_j^n}{\Delta t_n}
+c\frac{U_j^n-U_{j-1}^n}{h}=0.$$
Отсюда
$$U_j^{n+1}=(1-C_n)U_j^n+C_nU_{j-1}^n,
\qquad C_n=\frac{c\Delta t_n}{h}.$$
Число Куранта показывает, какую долю межузлового расстояния примесь проходит за один шаг. При $0\le C_n\le1$ новое значение лежит между двумя старыми, поэтому схема не создаёт новых экстремумов. При фиксированном $C$ её порядок на гладком решении — первый: пространственная и временная ошибки имеют порядок $h$ и $\Delta t$.

### Схема Лакса–Вендроффа

Разложим гладкое решение по времени до второго порядка. Из уравнения следуют $u_t=-cu_x$ и $u_{tt}=c^2u_{xx}$, поэтому
$$u(x,t+\Delta t)=u-c\Delta t\,u_x+
\frac{c^2\Delta t^2}{2}u_{xx}+O(\Delta t^3).$$
Заменяя пространственные производные центральными разностями, получаем
$$U_j^{n+1}=U_j^n-\frac{C_n}{2}(U_{j+1}^n-U_{j-1}^n)
+\frac{C_n^2}{2}(U_{j+1}^n-2U_j^n+U_{j-1}^n).$$
На гладком решении это схема второго порядка по времени и пространству. Её удобно также записать через веса трёх узлов:
$$U_j^{n+1}=\frac{C_n(C_n+1)}2 U_{j-1}^n
+(1-C_n^2)U_j^n+\frac{C_n(C_n-1)}2 U_{j+1}^n.$$
Веса в сумме дают единицу, но при $0<C_n<1$ последний вес отрицателен. Поэтому свойство сохранения диапазона, которое было у схемы против потока, здесь уже не следует из формулы.

Основной шаг выбираем как $\Delta t=Ch$, поскольку $c=1$. При сгущении сетки уменьшаем оба шага и сохраняем заданное $C$. Если нужный момент времени не кратен $\Delta t$, сокращаем последний шаг и пересчитываем все веса по фактическому $C_n$. Например, до $t=1$ при $N=50$, $C=0.8$ требуются 62 полных шага и один шаг с $C_n=0.4$.

In [ ]:
def smooth(x):
    return 1.0 + 0.5 * np.sin(2 * np.pi * x)


def pulse(x):
    return ((x >= 0.2) & (x < 0.4)).astype(float)


PROFILES = {'smooth': smooth, 'pulse': pulse}


def exact(x, t, profile):
    """Точный сдвиг по кольцу при c=1; значения относятся к узлам."""
    shifted = np.remainder(x - t, 1.0)
    if profile == 'pulse':
        # При попадании в разрыв восстанавливаем [0.2, 0.4): округление
        # координаты не должно менять значение с 0 на 1 или обратно.
        tolerance = 32 * np.finfo(float).eps * max(1.0, abs(t))
        for edge in [0.0, 0.2, 0.4, 1.0]:
            shifted = np.where(abs(shifted - edge) < tolerance,
                               edge % 1.0, shifted)
    return PROFILES[profile](shifted)


def upwind(u, C):
    """Один шаг схемы против потока; np.roll(u, 1) даёт левого соседа."""
    return (1 - C) * u + C * np.roll(u, 1)


def lax_wendroff(u, C):
    """Один шаг схемы Лакса–Вендроффа по значениям старого слоя."""
    left = np.roll(u, 1)
    right = np.roll(u, -1)
    return C * (C + 1) / 2 * left + (1 - C**2) * u + C * (C - 1) / 2 * right


METHODS = {'upwind': upwind, 'LW': lax_wendroff}


def simulate(N, C, t_end, profile='smooth'):
    """Расчёт обеими схемами до t_end с историей профиля, ошибки и массы."""
    if N <= 0 or int(N) != N or C <= 0 or t_end <= 0:
        raise ValueError('Требуются целое N > 0, C > 0 и t_end > 0')
    initial_profile = PROFILES[profile]
    x = np.arange(N) / N
    h = 1 / N
    dt = C * h

    # Число полных шагов и остаток определяем без накопления времени в цикле.
    n_full = int(np.floor(t_end / dt))
    remainder = t_end - n_full * dt
    tolerance = 32 * np.finfo(float).eps * max(1.0, t_end)
    if abs(remainder - dt) <= tolerance:
        n_full += 1
        remainder = 0.0
    elif abs(remainder) <= tolerance:
        remainder = 0.0

    times = np.arange(n_full + 1) * dt
    courant_steps = np.full(n_full, C)
    if remainder > 0:
        times = np.r_[times, t_end]
        courant_steps = np.r_[courant_steps, remainder / h]
    times[-1] = t_end

    result = dict(x=x, times=times, Cs=courant_steps, C=C, N=N, profile=profile)
    references = np.array([exact(x, t, profile) for t in times])
    for name, step in METHODS.items():
        history = np.empty((len(times), N))
        history[0] = initial_profile(x)
        for k, current_C in enumerate(courant_steps):
            history[k + 1] = step(history[k], current_C)
            if not np.all(np.isfinite(history[k + 1])):
                raise ArithmeticError(f'Переполнение при t={times[k + 1]:g}')
        result[name] = dict(
            u=history,
            error=h * np.sum(abs(history - references), axis=1),
            mass=h * history.sum(axis=1),
            minimum=history.min(axis=1),
            maximum=history.max(axis=1),
        )
    return result

Для каждого временного слоя вычисляем среднюю абсолютную ошибку и количество примеси:
$$E_1(t_n)=h\sum_{j=0}^{N-1}\left|U_j^n-u(x_j,t_n)\right|,
\qquad M_h(t_n)=h\sum_{j=0}^{N-1}U_j^n.$$
Так как длина области равна единице и $h=1/N$, первая формула совпадает с обычным средним абсолютных ошибок по узлам. Ошибку всегда считаем относительно точного решения в тот же физический момент времени.

Для прямоугольного импульса дополнительно определяем $u_{\min}(t_n)=\min_j U_j^n$ и $u_{\max}(t_n)=\max_j U_j^n$. Точное решение имеет минимум 0 и максимум 1. Отрицательные значения и превышения единицы в численном решении сохраняем: по ним оценивается нарушение физических границ концентрации.

Начальное количество примеси равно 1 для синусоиды и 0.2 для прямоугольника. Все выбранные $N$ кратны пяти, поэтому интервал $[0.2,0.4)$ содержит ровно $N/5$ узлов и его начальная дискретная масса также равна 0.2.

## 2. Гладкий профиль: один и пять оборотов

Зададим $u_0(x)=1+\tfrac12\sin(2\pi x)$ и $C=0.8$. Проведём расчёты на сетках $N=50$, $100$, $200$ до $t=1$ и $t=5$. В эти моменты точная синусоида возвращается в исходное положение: любые различия между профилями связаны с численным методом.

На рисунке каждая строка соответствует одной сетке, левый столбец — одному обороту, правый — пяти. Синяя сплошная линия показывает схему против потока, оранжевая штриховая — схему Лакса–Вендроффа, тонкий чёрный пунктир — точное решение.

In [ ]:
smooth_sizes = [50, 100, 200]
smooth_times = [1.0, 5.0]
smooth_runs = {(N, t): simulate(N, 0.8, t)
               for N in smooth_sizes for t in smooth_times}

fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True, sharey=True,
                         layout='constrained')
for row, N in enumerate(smooth_sizes):
    for col, t in enumerate(smooth_times):
        ax = axes[row, col]
        run = smooth_runs[N, t]
        for method in METHODS:
            plot_numerical(ax, run['x'], run[method]['u'][-1], method)
        plot_reference(ax, t, 'smooth')
        ax.set(xlim=(0, 1), ylim=(0.44, 1.56), xlabel='$x$', ylabel='$u$',
               title=f'N = {N}, t = {t:g}')
fig.legend(*axes[0, 0].get_legend_handles_labels(),
           loc='outside upper center', ncol=3, fontsize=10)
plt.show()

rows = []
for t in smooth_times:
    previous = {}
    for N in smooth_sizes:
        values = []
        for method in METHODS:
            error = smooth_runs[N, t][method]['error'][-1]
            order = ('—' if method not in previous else
                     f'{np.log2(previous[method] / error):.3f}')
            values.extend([f'{error:.5e}', order])
            previous[method] = error
        rows.append([f'{t:g}', N] + values)
table(['t', 'N', '$E_1$, против потока', '$p_N$',
       '$E_1$, Лакса–Вендроффа', '$p_N$'], rows)

На грубой сетке после пяти оборотов схема против потока заметно уменьшает амплитуду синусоиды. У схемы Лакса–Вендроффа профиль остаётся близким к точному, однако небольшая ошибка сохраняется и у неё. При $N=200$ ошибки после пяти оборотов составляют соответственно $0.0299$ и $0.000592$ — во втором случае примерно в 51 раз меньше.

В таблице наблюдаемый порядок относится к переходу от предыдущей сетки к текущей:
$$p_N=\log_2\frac{E_1(N/2)}{E_1(N)}.$$
Если $E_1\approx Ah^p$, то удвоение $N$ уменьшает ошибку в $2^p$ раз. Для схемы против потока после одного оборота получаем $p_N\approx0.99$, а для схемы Лакса–Вендроффа — около 2. После пяти оборотов порядок первой схемы составляет 0.87 и 0.93: на этих сетках накопленное затухание ещё заметно отклоняется от ведущего линейного по $h$ приближения. При сгущении значение приближается к единице.

Покажем ту же зависимость в логарифмических координатах. Вспомогательные прямые пропорциональны $h$ и $h^2$; их положение по вертикали произвольно, сравниваются наклоны.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, layout='constrained')
hs = 1 / np.array(smooth_sizes)
for ax, t in zip(axes, smooth_times):
    for method, marker, power in [('upwind', 'o', 1), ('LW', 's', 2)]:
        errors = np.array([smooth_runs[N, t][method]['error'][-1] for N in smooth_sizes])
        ax.loglog(hs, errors, marker=marker, **STYLES[method], label=LABELS[method])
        guide = 0.65 * errors[-1] * (hs / hs[-1])**power
        ax.loglog(hs, guide, color='0.4', ls=':' if power == 1 else '-.',
                  lw=1.2, label='$h$' if power == 1 else '$h^2$')
    ax.set(xlabel='Шаг сетки h', ylabel='$E_1$', title=f't = {t:g}, C = 0.8')
    ax.set_xticks(hs, ['1/50', '1/100', '1/200'])
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.legend(fontsize=9)
plt.show()

### Потеря амплитуды и ошибка положения

Синусоида позволяет разделить две причины ошибки. Если подставить в разностную схему гармонику $e^{ij\theta}$, один шаг умножает её на коэффициент
$$G_{\mathrm{up}}(C,\theta)=1-C+Ce^{-i\theta},$$
$$G_{\mathrm{LW}}(C,\theta)=1-iC\sin\theta+C^2(\cos\theta-1).$$
Для нашего начального профиля $\theta=2\pi/N$. Модуль произведения $P=\prod_nG(C_n,\theta)$ определяет изменение амплитуды, а его аргумент — сдвиг фазы. Поэтому результат можно вычислить непосредственно:
$$U_j=1+\frac12\operatorname{Im}\bigl(e^{i2\pi x_j}P\bigr).$$
Это даёт проверку пошагового решения и объясняет, какая ошибка в нём преобладает.

При $t=5$ точный множитель $e^{-i2\pi t}$ равен единице. Численная амплитуда равна $|P|/2$, фазовая ошибка — $\delta\varphi=\arg P$. Соответствующее смещение профиля относительно точного составляет $\delta x=-\delta\varphi/(2\pi)$: положительный знак означает опережение по направлению переноса, отрицательный — отставание. Рассчитаем эти величины для $N=100$.

In [ ]:
harmonic_rows = []
run = smooth_runs[100, 5.0]
theta = 2 * np.pi / run['N']
for method in METHODS:
    Cs = run['Cs']
    if method == 'upwind':
        factors = 1 - Cs + Cs * np.exp(-1j * theta)
    else:
        factors = 1 - 1j * Cs * np.sin(theta) + Cs**2 * (np.cos(theta) - 1)
    product = np.prod(factors)
    predicted = 1 + 0.5 * np.imag(np.exp(1j * 2 * np.pi * run['x']) * product)
    difference = np.max(abs(predicted - run[method]['u'][-1]))
    phase_error = np.angle(product)
    harmonic_rows.append([
        LABELS[method], f'{0.5 * abs(product):.6f}', f'{phase_error:+.5f}',
        f'{-phase_error / (2 * np.pi):+.6f}', f'{difference:.2e}',
    ])
    assert difference < 1e-12
table(['Схема', 'Амплитуда', r'$\delta\varphi$, рад', r'$\delta x$',
       'Разность расчётов, максимум'], harmonic_rows)

Схема против потока уменьшила амплитуду с 0.5 до 0.4104, то есть примерно на 18%. Смещение профиля при этом мало: около $+0.000395$. У схемы Лакса–Вендроффа амплитуда равна 0.49986, а профиль отстаёт примерно на 0.00118. Таким образом, первая схема прежде всего гасит синусоиду; у второй на этом опыте заметнее ошибка фазы. Пошаговый расчёт и формула через $P$ совпадают с точностью порядка $10^{-14}$.

### Сохраняется ли количество примеси

Обе схемы допускают запись через разность потоков:
$$U_j^{n+1}=U_j^n-\frac{\Delta t_n}{h}
\left(F_{j+1/2}^n-F_{j-1/2}^n\right),$$
где
$$F_{j+1/2}^{\mathrm{up}}=cU_j^n,\qquad
F_{j+1/2}^{\mathrm{LW}}=\frac c2(U_j^n+U_{j+1}^n)
-\frac{cC_n}{2}(U_{j+1}^n-U_j^n).$$
При суммировании по кольцу каждый поток входит один раз со знаком плюс и один раз со знаком минус. Поэтому
$$M_h^{n+1}-M_h^n=-\Delta t_n\sum_j
\left(F_{j+1/2}^n-F_{j-1/2}^n\right)=0.$$
Это свойство разностных формул выполняется и для сокращённого последнего шага. В вычислениях возможны лишь отклонения из-за округления. Покажем $\Delta M_h(t)=M_h(t)-M_h(0)$ на всех трёх сетках и сравним их с ошибкой профиля после пяти оборотов.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7), sharey=True, layout='constrained')
mass_rows = []
for ax, N in zip(axes, smooth_sizes):
    run = smooth_runs[N, 5.0]
    for method in METHODS:
        data = run[method]
        drift = data['mass'] - data['mass'][0]
        ax.plot(run['times'], drift, **STYLES[method], label=LABELS[method])
        mass_rows.append([N, LABELS[method], f'{np.max(abs(drift)):.2e}',
                          f"{data['error'][-1]:.5e}"])
    ax.set(xlabel='t', ylabel=r'$\Delta M_h$', title=f'N = {N}, C = 0.8')
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useOffset=False)
fig.legend(*axes[0].get_legend_handles_labels(),
           loc='outside upper center', ncol=2, fontsize=10)
plt.show()
table(['N', 'Схема', 'Макс. изменение массы на [0, 5]', '$E_1(5)$'], mass_rows)

Масса на всех сетках остаётся равной 1 с погрешностью округления. При этом, например, для $N=100$ ошибка распределения достигает $0.0570$ у схемы против потока и $0.00237$ у схемы Лакса–Вендроффа. Сохранение количества примеси не означает, что её распределение найдено правильно: уменьшение концентрации в одних местах может компенсироваться увеличением в других. У синусоиды любое изменение амплитуды при неизменном среднем значении вообще не меняет массу.

## 3. Прямоугольный импульс

Зададим
$$u_0(x)=\begin{cases}1,&0.2\le x<0.4,\\0,&\text{в остальных точках.}\end{cases}$$
Точный профиль движется вправо, сохраняя два скачка и высоту 1. Его левый край — задний фронт, правый — передний. После перехода через $x=1$ импульс продолжает движение от $x=0$.

Сначала сравним $C=0.5$, $0.8$, $1$ на одной сетке $N=200$. Первый рисунок показывает весь профиль после пяти оборотов. На втором увеличены оба фронта после одного оборота: ширина каждого окна равна 0.20, вертикальный масштаб во всех панелях одинаков. Численные узловые значения изображены ступенчато; при вычислении ошибок используются сами узлы.

In [ ]:
stable = {C: simulate(200, C, 5.0, 'pulse') for C in [0.5, 0.8, 1.0]}
front_runs = {C: simulate(200, C, 1.0, 'pulse') for C in stable}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True, layout='constrained')
for ax, (C, run) in zip(axes, stable.items()):
    for method in METHODS:
        plot_numerical(ax, run['x'], run[method]['u'][-1], method, pulse_profile=True)
    plot_reference(ax, 5.0, 'pulse')
    levels(ax)
    ax.set(xlim=(0, 1), ylim=(-0.37, 1.32), xlabel='$x$', ylabel='$u$',
           title=f'C = {C:g}, t = 5, N = 200')
fig.legend(*axes[0].get_legend_handles_labels(),
           loc='outside upper center', ncol=3, fontsize=10)
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True, layout='constrained')
fronts = [((0.1, 0.3), 'Задний фронт'), ((0.3, 0.5), 'Передний фронт')]
for col, (C, run) in enumerate(front_runs.items()):
    for row, (window, name) in enumerate(fronts):
        ax = axes[row, col]
        for method in METHODS:
            plot_numerical(ax, run['x'], run[method]['u'][-1], method, pulse_profile=True)
        plot_reference(ax, 1.0, 'pulse')
        levels(ax)
        ax.set(xlim=window, ylim=(-0.3, 1.3), xlabel='$x$', ylabel='$u$',
               title=f'{name}, C = {C:g}, t = 1')
        ticks = np.linspace(*window, 5)
        ax.set_xticks(ticks, [f'{value:.2f}' for value in ticks])
fig.legend(*axes[0, 0].get_legend_handles_labels(),
           loc='outside upper center', ncol=3, fontsize=10)
plt.show()

### Откуда берутся размытие и колебания

Схема против потока при $0<C<1$ на каждом шаге усредняет два соседних значения. Поэтому фронт постепенно растягивается, а высота импульса уменьшается. При $C=0.5$ через пять оборотов максимум равен 0.629; при $C=0.8$ он выше — 0.843. Во всём расчёте концентрация остаётся в пределах $[0,1]$.

Чтобы оценить причину размытия, разложим разностное уравнение для гладкой функции:
$$u_t+c u_x=-\frac{\Delta t}{2}u_{tt}+\frac{ch}{2}u_{xx}
+O(\Delta t^2+h^2).$$
С учётом $u_{tt}=c^2u_{xx}$ и $\Delta t=Ch/c$ ведущий член ошибки имеет вид диффузии:
$$u_t+c u_x\approx D_{\mathrm{num}}u_{xx},\qquad
D_{\mathrm{num}}=\frac{ch}{2}(1-C).$$
При $N=200$ это $D_{\mathrm{num}}=0.00125$ для $C=0.5$ и $0.0005$ для $C=0.8$: в первом случае численная диффузия в 2.5 раза сильнее. В исходном уравнении такого процесса нет. Разложение неприменимо в самой точке разрыва, но объясняет, почему после размазывания фронта его ширина продолжает расти и почему увеличение $C$ уменьшает этот эффект.

Схема Лакса–Вендроффа сохраняет более резкий переход, но создаёт колебания. Причину видно уже за первый шаг. В узле непосредственно слева от заднего фронта значения трёх соседей равны $(0,0,1)$, поэтому
$$U_j^1=\frac{C(C-1)}2=-\frac{C(1-C)}2<0.$$
В последнем единичном узле перед передним фронтом тройка равна $(1,1,0)$, откуда
$$U_j^1=1+\frac{C(1-C)}2>1.$$
Таким образом, при $C=0.5$ сразу возникают значения $-0.125$ и $1.125$, а при $C=0.8$ — $-0.08$ и $1.08$. Эти выбросы следуют из самих весов схемы.

На гладком профиле отрицательный вес помогает компенсировать ведущую диффузионную ошибку и получить второй порядок. На разрыве производные, использованные при выводе схемы, не существуют. Кроме того, разрыв содержит множество пространственных гармоник, для которых численный сдвиг фазы различается. Их наложение даёт колебания около фронтов. Устойчивость при $C\le1$ ограничивает рост нормы решения, но не гарантирует неотрицательность каждого узлового значения.

### Что меняется при сгущении сетки

Повторим расчёт до $t=5$ при $N=50$, $100$, $200$, $400$, сохраняя $C$. Для $C=0.5$ и $0.8$ приведём $E_1$, минимум и максимум, а также наблюдаемый порядок уменьшения $E_1$. Случай $C=1$ рассмотрим отдельно.

In [ ]:
pulse_sizes = [50, 100, 200, 400]
pulse_grid = {}
rows = []
for C in [0.5, 0.8]:
    previous = {}
    for N in pulse_sizes:
        run = stable[C] if N == 200 else simulate(N, C, 5.0, 'pulse')
        pulse_grid[N, C] = run
        for method in METHODS:
            data = run[method]
            error = data['error'][-1]
            order = ('—' if method not in previous else
                     f'{np.log2(previous[method] / error):.3f}')
            previous[method] = error
            rows.append([f'{C:g}', N, LABELS[method], f'{error:.5f}', order,
                         f"{data['minimum'][-1]:+.3e}", f"{data['maximum'][-1]:.5f}"])
table(['C', 'N', 'Схема', '$E_1(5)$', '$p_N$', 'Минимум', 'Максимум'], rows)

fig, axes = plt.subplots(3, 2, figsize=(11, 9), sharex=True, sharey='row',
                         layout='constrained')
hs = 1 / np.array(pulse_sizes)
quantities = [('error', '$E_1(5)$'), ('minimum', 'Минимум u'), ('maximum', 'Максимум u')]
for col, C in enumerate([0.5, 0.8]):
    for row, (key, label) in enumerate(quantities):
        ax = axes[row, col]
        for method, marker in [('upwind', 'o'), ('LW', 's')]:
            values = [pulse_grid[N, C][method][key][-1] for N in pulse_sizes]
            ax.plot(hs, values, marker=marker, **STYLES[method], label=LABELS[method])
        ax.set_xscale('log', base=2)
        ax.set_xticks(hs, ['1/50', '1/100', '1/200', '1/400'])
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set(ylabel=label, xlabel='Шаг сетки h' if row == 2 else None)
        if row == 0:
            ax.set_yscale('log')
            ax.set_title(f'C = {C:g}, t = 5')
        else:
            ax.axhline(0 if row == 1 else 1, color='0.4', ls=':', lw=1)
        if row == 1:
            ax.set_ylim(-0.36, 0.1)
        if row == 2:
            ax.set_ylim(0.3, 1.3)
fig.legend(*axes[0, 0].get_legend_handles_labels(),
           loc='outside upper center', ncol=2, fontsize=10)
plt.show()

У обеих схем интегральная ошибка уменьшается, однако порядки 1 и 2, полученные на синусоиде, на этих сетках не воспроизводятся. Например, при $C=0.8$ удвоение $N$ с 200 до 400 уменьшает $E_1$ лишь в 1.41 раза у схемы против потока и в 1.46 раза у схемы Лакса–Вендроффа. Соответствующие порядки равны примерно 0.50 и 0.55. Это измеренные значения для выбранных сеток и времени, а не универсальные порядки на любом разрывном профиле.

Для схемы против потока при $C=0.8$ максимум возрастает с 0.520 на сетке $N=50$ до 0.955 на сетке $N=400$: импульс лучше сохраняет высоту. Для схемы Лакса–Вендроффа на мелкой сетке получаем минимум $-0.232$ и максимум $1.232$. Интегральная ошибка у неё меньше, но локальные значения остаются нефизическими. Уменьшение площади ошибки допускает сохранение высоких выбросов в более узкой области. Поэтому убывание $E_1$ само по себе не означает уменьшения максимальной ошибки.

Зависимость экстремумов от $N$ у схемы Лакса–Вендроффа немонотонна, особенно при $C=0.5$. При изменении сетки меняются расположение колебательных пиков относительно узлов и наложение возмущений от двух фронтов. Оценивать качество только по одному значению максимума на одной сетке было бы недостаточно.

### Почему $C=1$ требует отдельного рассмотрения

При подстановке $C=1$ обе разностные формулы превращаются в
$$U_j^{n+1}=U_{j-1}^n.$$
Точная характеристика за шаг также проходит ровно одно межузловое расстояние. Поэтому на каждом временном слое перенос уже заданного массива точен: за $N$ шагов профиль совершает оборот и возвращается к начальному. Это справедливо и для разрывного профиля. Проверим пять оборотов на каждой сетке.

In [ ]:
rows = []
for N in pulse_sizes:
    run = stable[1.0] if N == 200 else simulate(N, 1.0, 5.0, 'pulse')
    for method in METHODS:
        data = run[method]
        assert np.array_equal(data['u'][-1], data['u'][0])
        assert np.max(data['error']) == 0
        rows.append([N, LABELS[method], f"{data['error'][-1]:.1e}",
                     f"{data['minimum'][-1]:g}", f"{data['maximum'][-1]:g}"])
table(['N', 'Схема', '$E_1(5)$', 'Минимум', 'Максимум'], rows)

Нулевая ошибка при $C=1$ связана с точным совпадением переноса и сдвига на один узел. По ней нельзя оценивать обычный порядок сходимости: отношение нулевых ошибок не определено. Если последний шаг приходится сократить, его число Куранта становится меньше единицы и точность сдвига на этом шаге теряется. Кроме того, точный перенос массива не восстанавливает детали начального профиля, которые не были разрешены сеткой.

### Как ошибки накапливаются во времени

До сих пор сравнивались отдельные моменты. Теперь проследим минимум, максимум и $E_1$ на всём интервале $0\le t\le5$ при $N=200$. Число Куранта меняется по столбцам, измеряемая величина — по строкам. Масштабы в пределах каждой строки одинаковы, чтобы различия между $C$ были видны без пересчёта осей.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 8.5), sharex=True, sharey='row',
                         layout='constrained')
quantities = [
    ('minimum', 'Минимум u', (-0.4, 0.08)),
    ('maximum', 'Максимум u', (0.25, 1.35)),
    ('error', '$E_1$', (-0.01, 0.23)),
]
for col, (C, run) in enumerate(stable.items()):
    for row, (key, label, limits) in enumerate(quantities):
        ax = axes[row, col]
        for method in METHODS:
            ax.plot(run['times'], run[method][key],
                    **STYLES[method], label=LABELS[method])
        ax.set(ylim=limits, ylabel=label, xlabel='t' if row == 2 else None)
        if row == 0:
            ax.set_title(f'C = {C:g}, N = 200')
            ax.axhline(0, color='0.4', ls=':', lw=1, zorder=1)
        if row == 1:
            ax.axhline(1, color='0.4', ls=':', lw=1, zorder=1)
fig.legend(*axes[0, 0].get_legend_handles_labels(),
           loc='outside upper center', ncol=2, fontsize=10)
plt.show()

rows = []
for C, run in stable.items():
    for method in METHODS:
        mass = run[method]['mass']
        rows.append([f'{C:g}', LABELS[method], f'{np.max(abs(mass - mass[0])):.2e}'])
table(['C', 'Схема', 'Макс. изменение массы на [0, 5]'], rows)

У схемы против потока максимум постепенно снижается; при $C=0.5$ это происходит быстрее. Минимум остаётся неотрицательным. У схемы Лакса–Вендроффа экстремумы выходят за $[0,1]$ сразу и затем колеблются, но не демонстрируют быстрого нарастания по модулю. При $C=1$ минимум равен 0, максимум — 1, а ошибка — нулю на всех временных слоях.

Небольшая пилообразность $E_1(t)$ связана в том числе с прохождением точных фронтов через узлы: значение точного решения в узле меняется скачком, тогда как численный фронт размазан. Масса во всех трёх опытах сохраняется с погрешностью округления. Значит, и размытие, и паразитные колебания могут сопровождаться правильным общим количеством примеси.

## 4. Что происходит при $C=1.1$

За один шаг примесь теперь проходит $1.1h$, а явная разностная формула использует только ближайших соседей. Нарушается условие Куранта–Фридрихса–Леви: численная область зависимости не охватывает физическую. Для выбранных схем неустойчивость можно установить непосредственно по коэффициентам усиления:
$$|G_{\mathrm{up}}|^2=1-4C(1-C)\sin^2(\theta/2),$$
$$|G_{\mathrm{LW}}|^2=1-4C^2(1-C^2)\sin^4(\theta/2).$$
При $0\le C\le1$ каждый модуль не превосходит единицы. При $C>1$ непостоянные гармоники растут. Для предельной сеточной частоты $\theta=\pi$ при $C=1.1$ модули равны 1.2 и 1.42. Это множители за один шаг; многократное повторение шага приводит к экспоненциальному усилению.

Возьмём прямоугольный профиль на сетке $N=100$ и выполним 50 шагов до $t=0.55$. У этого прямоугольника вклад именно частоты $\theta=\pi$ равен нулю, поскольку число единичных узлов чётное, но соседние высокие частоты присутствуют и также неустойчивы. Дополнительное возмущение для появления роста не требуется.

Графики неустойчивого расчёта показаны отдельно. У максимума и ошибки ось ординат логарифмическая. Для отрицательного минимума применена симметричная логарифмическая шкала: она сохраняет знак и позволяет показать начало роста от нуля.

In [ ]:
unstable = simulate(100, 1.1, 0.55, 'pulse')
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), layout='constrained')
for ax, key, label in zip(axes, ['minimum', 'maximum', 'error'],
                           ['Минимум u', 'Максимум u', '$E_1$']):
    for method in METHODS:
        values = unstable[method][key]
        if key == 'error':
            values = np.ma.masked_less_equal(values, 0)  # log(0) не определён.
        ax.plot(unstable['times'], values, **STYLES[method], label=LABELS[method])
    ax.set(xlabel='t', ylabel=label, title=f'C = 1.1, N = 100')
    if key == 'minimum':
        ax.set_yscale('symlog', linthresh=0.1)
    else:
        ax.set_yscale('log')
fig.legend(*axes[0].get_legend_handles_labels(),
           loc='outside upper center', ncol=2, fontsize=10)
plt.show()

rows = []
for method in METHODS:
    data = unstable[method]
    rows.append([
        LABELS[method], f"{data['minimum'][-1]:.3e}",
        f"{data['maximum'][-1]:.3e}", f"{data['error'][-1]:.3e}",
        f"{np.max(abs(data['mass'] - data['mass'][0])):.2e}",
    ])
table(['Схема', 'Минимум при t=0.55', 'Максимум при t=0.55',
       '$E_1(0.55)$', 'Макс. изменение массы'], rows)

К $t=0.55$ схема против потока даёт значения примерно от $-956$ до $957$, а схема Лакса–Вендроффа — от $-2.34\cdot10^6$ до $2.34\cdot10^6$. Ошибка достигает соответственно 91 и $4.10\cdot10^5$ при начальной высоте импульса 1. Это развитие неустойчивости; ограниченные колебания на устойчивых расчётах такого роста не показывали.

Даже здесь вычисленная масса остаётся близкой к 0.2. Причина в том, что постоянная гармоника имеет $G(0)=1$, а растут гармоники с нулевым средним. Большие положительные и отрицательные значения пока компенсируются при суммировании. При дальнейшем росте ухудшится и точность вычисления массы из-за вычитания больших близких величин. Поэтому сохранение массы не может служить самостоятельным критерием ни точности, ни устойчивости.

## 5. Перенос в движении

Анимация показывает пять оборотов прямоугольного импульса на сетке $N=200$ при $C=0.5$, $0.8$, $1$. Во всех панелях одинаковы время и масштаб концентрации. При прохождении границы $x=1$ профиль появляется у $x=0$ — это продолжение движения по кольцу.

У схемы против потока постепенно сглаживаются оба фронта. У схемы Лакса–Вендроффа вместе с импульсом движутся колебания, а при $C=1$ оба численных профиля совпадают с точным. Пределы вертикальной оси фиксированы, поэтому потеря высоты и выбросы сравниваются в одном масштабе.

In [ ]:
# Интервал 0.02 кратен шагам всех трёх расчётов: кадры не интерполируются.
frame_times = np.arange(251) * 0.02
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), sharey=True)
artists = []
for ax, (C, run) in zip(axes, stable.items()):
    numerical_lines = [plot_numerical(ax, run['x'], run[method]['u'][0],
                                     method, pulse_profile=True) for method in METHODS]
    reference_line = plot_reference(ax, 0, 'pulse')
    levels(ax)
    ax.set(xlim=(0, 1), ylim=(-0.4, 1.35), xlabel='$x$', ylabel='$u$',
           title=f'N = 200, C = {C:g}')
    artists.append((run, numerical_lines, reference_line))
fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=3, fontsize=10)
time_label = fig.suptitle('Перенос прямоугольного импульса; t = 0.00')
fig.tight_layout(rect=(0, 0.12, 1, 0.92))


def update(frame):
    t = frame_times[frame]
    for run, numerical_lines, reference_line in artists:
        index = int(round(t / (run['C'] / run['N'])))
        assert abs(run['times'][index] - t) < 1e-12
        for method, line in zip(METHODS, numerical_lines):
            u = run[method]['u'][index]
            line.set_ydata(np.r_[u, u[0]])
        xx, yy = exact_pulse_line(t)
        reference_line.set_data(xx, yy)
    time_label.set_text(f'Перенос прямоугольного импульса; t = {t:.2f}')


with plt.rc_context({'savefig.dpi': 90, 'animation.frame_format': 'jpeg'}):
    animation = FuncAnimation(fig, update, frames=len(frame_times),
                              interval=1000 / 15, blit=False)
    animation_html = animation.to_jshtml(fps=15, default_mode='once')
plt.close(fig)
display(HTML(animation_html))

## 6. Результаты

На гладкой синусоиде при $C=0.8$ получен первый порядок сходимости для схемы против потока и второй для схемы Лакса–Вендроффа. После пяти оборотов на сетке $N=200$ ошибки $E_1$ равны $0.0299$ и $0.000592$. Разбор одной гармоники показывает, что у первой схемы преобладает затухание амплитуды, а у второй в проведённом опыте заметнее сдвиг фазы.

На прямоугольном импульсе преимущество в интегральной ошибке сохраняется у схемы Лакса–Вендроффа, но её отрицательный вес создаёт значения вне $[0,1]$. При $C=0.8$, $N=400$, $t=5$ получены минимум $-0.232$ и максимум $1.232$, хотя $E_1$ снизилась до $0.0442$. Схема против потока на той же сетке даёт $E_1=0.0798$ и максимум 0.955, сохраняя неотрицательность. Таким образом, меньшая интегральная ошибка и соблюдение физических границ концентрации — разные свойства схемы.

При $C=1$ обе схемы точно сдвигают узловые значения; на всех рассмотренных сетках ошибка равна нулю. Это особое совпадение сеточного шага с перемещением за шаг времени. При $C=1.1$ обе схемы неустойчивы: уже до одного оборота амплитуда становится на несколько порядков больше начальной.

Количество примеси сохраняется благодаря консервативной форме обеих схем. Это необходимая проверка баланса, но она не выявляет ни размытие, ни паразитные колебания, ни рост неустойчивых гармоник с нулевым средним. Поэтому качество переноса оценивалось одновременно по массе, ошибке относительно точного профиля и экстремумам концентрации.

Материалы: [условие задания 6.1](https://github.com/yartmihant/Students6Year2026/blob/70c6a14acc8364f49825dd0c94515983eac69f96/README.md), [теоретический конспект](https://github.com/yartmihant/Students6Year2026/blob/70c6a14acc8364f49825dd0c94515983eac69f96/lesson6/lesson_06_theory.md), [демонстрационный ноутбук](https://github.com/yartmihant/Students6Year2026/blob/70c6a14acc8364f49825dd0c94515983eac69f96/lesson6/lesson_06_advection.ipynb).